In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import StackingClassifier

---
## <u>Generate Dataset</u>

In [6]:
X, y = make_classification(
    n_samples = 10000,
    n_classes = 3,
    n_features = 15,
    n_informative = 10,
    n_redundant = 2,
    random_state = 42
)

X = pd.DataFrame(X)

---
## <u>Train test split</u>

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

X_train.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
9254,3.360718,0.257881,1.914265,0.617164,-2.977548,2.075280,1.301235,1.436023,-1.478920,-1.532499,1.366877,-2.443176,1.398361,5.115375,-2.993145
1561,1.628488,0.233073,-3.374230,-1.026216,-0.896835,1.700875,1.005049,-0.075828,-2.087462,-1.733569,0.280852,-0.358967,-0.262196,-1.849775,0.066212
1670,-3.950286,-0.416480,-1.095017,2.162211,3.801133,0.683062,0.438400,-2.642000,0.389139,4.064865,4.129993,-3.857887,-3.240236,0.282855,-0.480733
6087,0.911172,0.122043,0.929496,0.809337,-0.808305,-1.341194,-1.525063,0.067394,-0.413805,-3.130134,1.240656,0.946087,2.381506,-1.419483,0.739948
6669,-1.634659,0.047966,0.238297,0.553826,2.115025,-0.158962,0.126237,-0.711588,0.503820,1.772115,0.363252,-0.387747,-0.503002,-1.066276,-0.204251


---
## <u>Define the baseline learners</u>

In [14]:
lgr_model = LogisticRegression(random_state = 42)
knn_classifier_model = KNeighborsClassifier()   
svc_model = SVC(random_state = 42)
dt_classifier_model = DecisionTreeClassifier(random_state = 42, max_depth = 4)

---
## <u>Create Stacking Classifier</u>

In [15]:
meta_model = LogisticRegression()

stacking_classifier = StackingClassifier(
    estimators = [                         # all the base classification learners
        ("lgr", lgr_model),
        ("knn_c", knn_classifier_model),
        ("svc", svc_model),
        ("dt_c", dt_classifier_model)
    ],
    final_estimator = meta_model,         # the final model which gets the predications of all the estimators
    cv=7                                  # numbers of folds for cross validation
)

---
## <u>Train Model</u>

In [16]:
stacking_classifier.fit(X_train, y_train)

,estimators,"[('lgr', ...), ('knn_c', ...), ...]"
,final_estimator,LogisticRegression()
,cv,7
,stack_method,'auto'
,n_jobs,None
,passthrough,False
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


---
## <u>Predict and Evaluate</u>

In [17]:
y_train_pred = stacking_classifier.predict(X_train)
y_test_pred = stacking_classifier.predict(X_test)

print("For Stacking Classifier (baseline) :-")
print("\nFor Training set :-")
print("accuracy :", accuracy_score(y_train, y_train_pred))
print("classification report :\n", classification_report(y_train, y_train_pred))

print("\nFor Test set :-")
print("accuracy :", accuracy_score(y_test, y_test_pred))
print("classification report :\n", classification_report(y_test, y_test_pred))

For Stacking Classifier (baseline) :-

For Training set :-
accuracy : 0.950375
classification report :
               precision    recall  f1-score   support

           0       0.95      0.94      0.94      2655
           1       0.95      0.96      0.96      2658
           2       0.95      0.96      0.95      2687

    accuracy                           0.95      8000
   macro avg       0.95      0.95      0.95      8000
weighted avg       0.95      0.95      0.95      8000


For Test set :-
accuracy : 0.9405
classification report :
               precision    recall  f1-score   support

           0       0.94      0.92      0.93       681
           1       0.94      0.95      0.94       669
           2       0.94      0.95      0.94       650

    accuracy                           0.94      2000
   macro avg       0.94      0.94      0.94      2000
weighted avg       0.94      0.94      0.94      2000

